# Weight-space programming, step 1: forward-pass probes of the JEPA adapter

Read-only probes for editing the trained JEPA adapter of `jepa-guided-diffusion` by hand (no training, no gradients), in the spirit of *Sparks of Weight-Space Programming* (Takehi et al., 2026). Known failure: on one image the JEPA-conditioned chain under-denoises (keeps 2.4× more of the input noise).

The adapter writes into ADM through four paths: feature offsets after encoder blocks 9, 12 and 15 (32, 16, 8 px) and an offset `glob` on ADM's time embedding, which is ADM's noise-level signal. Here each path is switched to the JEPA latents on its own, everything else at the null-latent value, and the one-step Tweedie estimate at t* is scored by PSNR and by the share of the input noise it keeps.

* **Images:** 40 DIV2K validation images (indices 16–55) that are **not** in v1's 16-image evaluation set, σ ∈ {0.1, 0.2}.
* **Exported for the weight analysis:** ADM's time embedding emb(t) for every t, the null `glob` offset, and the JEPA latents of every probe image.
* **Chains:** the 4 held-out cases where JEPA keeps the most noise relative to null also get full reverse chains.

Kaggle: GPU T4 ×2 (ADM on GPU 0, I-JEPA on GPU 1), internet on, outputs of `jepa-denoise-finetune` (v2) and `jepa-guided-diffusion` (v1) attached.

In [ ]:
import glob, io, json, math, os, random, re, subprocess, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.checkpoint import checkpoint
from PIL import Image
import pandas as pd

HELD_OUT = list(range(16, 56))   # DIV2K valid images outside v1's 16-image evaluation set
PROBE_SIGMAS = [0.1, 0.2]
N_CHAIN = 4                      # held-out cases with the largest JEPA − null kept noise that also get full reverse chains
N_EVAL = 16

assert torch.cuda.is_available(), "GPU required"
D0 = torch.device("cuda:0")
D1 = torch.device("cuda:1") if torch.cuda.device_count() > 1 else D0
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch.__version__)
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/guided_cache"), "outputs"
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/workspace/inputs")]
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
S = 256


def find_input(pattern):
    """First file matching the glob pattern under the input roots (Kaggle-attached outputs or $INPUT_DIR), else None."""
    for root in INPUT_ROOTS:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return hits[0]
    return None


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if size and have == size:
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def load_zip_images(path, short):
    with zipfile.ZipFile(path) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))

    def one(name):
        with zipfile.ZipFile(path) as z:
            im = Image.open(io.BytesIO(z.read(name))).convert("RGB")
        return torch.from_numpy(np.asarray(TF.resize(im, short)).copy()).permute(2, 0, 1).contiguous()

    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(one, names))


def fetch_image(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)

In [ ]:
DIV2K = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
VALID_IMGS = load_zip_images(download(DIV2K + "DIV2K_valid_HR.zip", f"{CACHE}/DIV2K_valid_HR.zip", size=448_993_893), S)
CROPS = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS])
EVAL_CLEAN = CROPS[:N_EVAL]   # only used by the adapter cell's sanity asserts
print(len(VALID_IMGS), "validation images; probing", len(HELD_OUT), "held-out ones")

In [ ]:
GD, IJ = f"{CACHE}/guided-diffusion", f"{CACHE}/ijepa"
for url, path in (("https://github.com/openai/guided-diffusion", GD), ("https://github.com/facebookresearch/ijepa", IJ)):
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, path], check=True)
sys.path[:0] = [GD, IJ]
from guided_diffusion.script_util import create_model
from guided_diffusion.nn import timestep_embedding
import guided_diffusion.unet as gd_unet

# ADM's attention blocks always use its own checkpoint function, which differentiates w.r.t. the block weights and
# fails when they are frozen; eps_pred() checkpoints whole blocks with torch.utils.checkpoint instead.
gd_unet.checkpoint = lambda func, inputs, params, flag: func(*inputs)
from src.models import vision_transformer as vt

adm = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                   attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                   use_scale_shift_norm=True)  # the README's flags for 256x256_diffusion_uncond.pt
ADM_CKPT = download("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                    f"{CACHE}/256x256_diffusion_uncond.pt", size=2_211_383_297)
adm.load_state_dict(torch.load(ADM_CKPT, map_location="cpu", weights_only=True))
adm.to(D0).eval().requires_grad_(False).convert_to_fp16()


def _sdpa_attention(self, x):  # same math as I-JEPA's attention, memory-efficient kernel
    B, n, C = x.shape
    q, k, v = self.qkv(x).reshape(B, n, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    x = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
    return self.proj(x.transpose(1, 2).reshape(B, n, C)), None


vt.Attention.forward = _sdpa_attention
found = [path for path in [find_input("jepa_denoise_vith14_fp16.pt")] if path]
assert found, "attach the output of the jepa-denoise-finetune notebook (version 2) as input data"
jepa = vt.vit_huge(patch_size=14)
jepa.load_state_dict(torch.load(found[0], map_location="cpu", weights_only=True)["target_encoder"])
jepa = jepa.half().to(D1).eval().requires_grad_(False)
JMEAN = torch.tensor([0.485, 0.456, 0.406], device=D1).view(1, 3, 1, 1)
JSTD = torch.tensor([0.229, 0.224, 0.225], device=D1).view(1, 3, 1, 1)


@torch.no_grad()
def jepa_latents(x):
    """x [B,3,S,S] in [0,1] -> [B,D,16,16]: LayerNormed target-encoder tokens of the 224 px resized image."""
    x = F.interpolate(x.to(D1).float(), size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
    with torch.autocast("cuda", dtype=torch.float16):
        h = jepa((x - JMEAN) / JSTD)
    h = F.layer_norm(h.float(), (h.size(-1),))
    return h.transpose(1, 2).reshape(len(x), h.size(-1), 16, 16).to(D0)

In [ ]:
@torch.no_grad()
def injection_points(res_list=(32, 16, 8)):
    """{ADM encoder block index: (resolution, channels)} for the first block at each resolution."""
    emb = adm.time_embed(timestep_embedding(torch.zeros(1, dtype=torch.long, device=D0), adm.model_channels))
    h, points = torch.zeros(1, 3, S, S, device=D0).type(adm.dtype), {}
    for i, m in enumerate(adm.input_blocks):
        h = m(h, emb)
        if h.shape[-1] in res_list and h.shape[-1] not in [r for r, _ in points.values()]:
            points[i] = (h.shape[-1], h.shape[1])
    return points


def _zero(m):
    nn.init.zeros_(m.weight)
    nn.init.zeros_(m.bias)
    return m


class JEPAAdapter(nn.Module):
    """JEPA latents [B,D,16,16] -> features added to ADM encoder blocks + an offset on ADM's time embedding."""

    def __init__(self, d, points, emb_dim):
        super().__init__()
        self.res = {str(i): r for i, (r, _) in points.items()}
        self.branches = nn.ModuleDict({str(i): nn.Sequential(
            nn.Conv2d(d, ch, 1), nn.GroupNorm(32, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU(),
            _zero(nn.Conv2d(ch, ch, 1))) for i, (r, ch) in points.items()})
        self.glob = nn.Sequential(nn.Linear(d, emb_dim), nn.SiLU(), _zero(nn.Linear(emb_dim, emb_dim)))

    def forward(self, c):
        feats = {}
        for i, branch in self.branches.items():
            r = self.res[i]
            feats[int(i)] = branch(c if r == c.shape[-1] else F.interpolate(c, size=(r, r), mode="bilinear", align_corners=False))
        return feats, self.glob(c.mean((2, 3)))


def eps_pred(x, t, cond=None, grad_ckpt=False):
    """ADM's UNetModel.forward (ε channels only) with adapter features added after the injection blocks."""
    emb = adm.time_embed(timestep_embedding(t, adm.model_channels))
    feats = {}
    if cond is not None:
        feats, g = cond
        emb = emb + g
    run = (lambda m, *a: checkpoint(m, *a, use_reentrant=False)) if grad_ckpt else (lambda m, *a: m(*a))
    hs, h = [], x.type(adm.dtype)
    for i, m in enumerate(adm.input_blocks):
        h = run(m, h, emb)
        if i in feats:
            h = h + feats[i].type(h.dtype)
        hs.append(h)
    h = run(adm.middle_block, h, emb)
    for m in adm.output_blocks:
        h = run(m, torch.cat([h, hs.pop()], 1), emb)
    return adm.out(h.type(x.dtype))[:, :3].float()


INJECT = injection_points()
adapter = JEPAAdapter(jepa.embed_dim, INJECT, adm.model_channels * 4).to(D0)
print("injection points {block: (px, channels)}:", INJECT, f"| adapter {sum(p.numel() for p in adapter.parameters()) / 1e6:.1f}M params")

BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)
ABAR_D = ABAR.float().to(D0)

with torch.no_grad():  # the re-implemented forward equals ADM's, and the untrained adapter changes nothing
    xt, tt = torch.randn(2, 3, S, S, device=D0), torch.tensor([10, 500], device=D0)
    ref = adm(xt, tt)[:, :3].float()
    assert torch.allclose(eps_pred(xt, tt), ref, atol=1e-3)
    assert torch.allclose(eps_pred(xt, tt, adapter(jepa_latents(EVAL_CLEAN[:2]))), ref, atol=1e-3)


found = find_input("jepa-guided-diffusion/**/jepa_adapter.pt")
assert found, "attach the output of the jepa-guided-diffusion notebook (version 1) as input data"
adapter.load_state_dict(torch.load(found, map_location="cpu", weights_only=True))
adapter.eval()


def t_for_sigma(sigma):  # pixel σ on [0, 1] -> step whose noise level matches (x in [-1, 1] doubles σ)
    return int((ABAR - 1 / (1 + (2 * sigma) ** 2)).abs().argmin())


def to01(z):
    return ((z + 1) / 2).clamp(0, 1).cpu()


with torch.no_grad():
    EMB = adm.time_embed(timestep_embedding(torch.arange(1000, device=D0), adm.model_channels)).float().cpu()
    F0, G0 = adapter(torch.zeros(1, jepa.embed_dim, 16, 16, device=D0))
torch.save({"emb": EMB, "g_null": G0.float().cpu()}, f"{OUT}/time_embedding.pt")
print("time embedding", tuple(EMB.shape), "rms", float(EMB.pow(2).mean().sqrt()), "| null glob offset rms", float(G0.pow(2).mean().sqrt()))

## Tweedie probes: which path makes the estimate keep noise?

In [ ]:
BLOCKS = sorted(INJECT)   # ADM encoder blocks that receive adapter features (9, 12, 15 for the 256x256 model)
VARIANTS = {"ADM": None, "null": (set(), False), "JEPA": (set(BLOCKS), True), "glob only": (set(), True),
            "spatial only": (set(BLOCKS), False), **{f"block {b} only": ({b}, False) for b in BLOCKS}}


def conds(c, names):
    """Batched adapter output for the named variants: paths in the spec use the JEPA latents c, the others c = 0."""
    fc, gc = adapter(c)
    f0, g0 = adapter(torch.zeros_like(c))
    feats = {i: torch.cat([torch.zeros_like(f0[i]) if VARIANTS[v] is None else (fc[i] if i in VARIANTS[v][0] else f0[i]) for v in names])
             for i in f0}
    emb = torch.cat([torch.zeros_like(g0) if VARIANTS[v] is None else (gc if VARIANTS[v][1] else g0) for v in names])
    return feats, emb


def scores_of(out, clean, noisy):
    """PSNR and the least-squares share of the input noise kept, per image in the batch."""
    noise = (noisy - clean).flatten()
    err = (out - clean[None]).flatten(1)
    psnr = 10 * torch.log10(1 / err.pow(2).mean(1))
    kept = (err @ noise) / (noise @ noise)
    return psnr.tolist(), kept.tolist()


rows, LATENTS, t_start = [], {}, time.time()
names = list(VARIANTS)
with torch.no_grad():
    for i in HELD_OUT:
        clean = CROPS[i]
        for j, s in enumerate(PROBE_SIGMAS):
            noisy = add_gaussian(clean, s, seed=50_000 + 100 * i + j)
            c = jepa_latents(noisy[None])
            LATENTS[(i, s)] = c.half().cpu()
            t0 = t_for_sigma(s)
            a = ABAR[t0].item()
            x = (a ** 0.5 * (2 * noisy.to(D0) - 1))[None].repeat(len(names), 1, 1, 1)
            eps = eps_pred(x, torch.full((len(names),), t0, device=D0, dtype=torch.long), conds(c, names))
            out = to01(((x - (1 - a) ** 0.5 * eps) / a ** 0.5).clamp(-1, 1))
            psnr, kept = scores_of(out, clean, noisy)
            rows += [dict(image=i, sigma=s, variant=v, psnr=p, kept=k) for v, p, k in zip(names, psnr, kept)]
probe = pd.DataFrame(rows)
probe.to_csv(f"{OUT}/probe_tweedie.csv", index=False)
torch.save({f"{i}|{s}": c for (i, s), c in LATENTS.items()}, f"{OUT}/latents.pt")
print(f"{len(LATENTS)} probe cases in {(time.time() - t_start) / 60:.1f} min")
wide = probe.pivot_table(index=["image", "sigma"], columns="variant", values=["psnr", "kept"])
for s in PROBE_SIGMAS:
    w = wide.xs(s, level="sigma")
    print(f"\nσ {s}: mean over {len(w)} images, difference to null")
    print(pd.DataFrame({"Δ PSNR (dB)": (w["psnr"].sub(w["psnr"]["null"], axis=0)).mean(),
                        "Δ kept": (w["kept"].sub(w["kept"]["null"], axis=0)).mean(),
                        "worst Δ kept": (w["kept"].sub(w["kept"]["null"], axis=0)).max()}).reindex(names).round(4).to_string())

## Full reverse chains for the worst held-out cases

In [ ]:
gap = (wide["kept"]["JEPA"] - wide["kept"]["null"]).sort_values(ascending=False)
CHAIN_CASES = list(gap.index[:N_CHAIN])
print("largest JEPA − null kept (Tweedie):", [(i, s, round(g, 4)) for (i, s), g in gap.head(N_CHAIN).items()])
CHAIN_VARIANTS = ["ADM", "null", "JEPA", "glob only", "spatial only"]


@torch.no_grad()
def chain(noisy, sigma, c, names, seed):
    """v1's reverse process from t* for several variants in one batch, same start and same noise draws."""
    cond, V, t0 = conds(c, names), len(names), t_for_sigma(sigma)
    gen = torch.Generator(device=D0).manual_seed(seed)
    x = (ABAR[t0].sqrt().float() * (2 * noisy.to(D0) - 1))[None].repeat(V, 1, 1, 1)
    for t in range(t0, -1, -1):
        a_t, beta_t = ABAR[t].item(), BETAS[t].item()
        a_prev = ABAR[t - 1].item() if t > 0 else 1.0
        eps = eps_pred(x, torch.full((V,), t, device=D0, dtype=torch.long), cond)
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        if t == 0:
            x = x0
            break
        mean = (a_prev ** 0.5 * beta_t / (1 - a_t)) * x0 + ((1 - beta_t) ** 0.5 * (1 - a_prev) / (1 - a_t)) * x
        x = mean + (beta_t * (1 - a_prev) / (1 - a_t)) ** 0.5 * torch.randn((1, 3, S, S), generator=gen, device=D0).expand_as(x)
    return to01(x)


chain_rows, t_start = [], time.time()
for i, s in CHAIN_CASES:
    clean = CROPS[i]
    noisy = add_gaussian(clean, s, seed=50_000 + 100 * i + PROBE_SIGMAS.index(s))
    out = chain(noisy, s, LATENTS[(i, s)].float().to(D0), CHAIN_VARIANTS, seed=i)
    psnr, kept = scores_of(out, clean, noisy)
    chain_rows += [dict(image=i, sigma=s, variant=v, psnr=p, kept=k) for v, p, k in zip(CHAIN_VARIANTS, psnr, kept)]
    Image.fromarray((torch.cat([clean, noisy, *out], 2).permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)).save(f"{OUT}/chain_{i}_{s}.png")
chains = pd.DataFrame(chain_rows)
chains.to_csv(f"{OUT}/probe_chain.csv", index=False)
print(f"chains in {(time.time() - t_start) / 60:.1f} min")
print(chains.pivot_table(index=["image", "sigma"], columns="variant", values=["psnr", "kept"]).reindex(columns=CHAIN_VARIANTS, level=1).round(3).to_string())
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)))